In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os


train_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 32x32
    transforms.RandomRotation(15),  # Random rotation augmentation
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images to 32x32
    transforms.ToTensor(),
])

# Load the dataset using ImageFolder - thanks god for ImageFolder <3
train_dataset = ImageFolder("/kaggle/input/q1-stage-3-2026/PlantVillage/train", transform=train_transform)
test_dataset = ImageFolder("/kaggle/input/q1-stage-3-2026/PlantVillage/test", transform=test_transform)


batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Classes: {class_names}")
print(f"Number of classes: {num_classes}")

# Display some sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

# Get a batch of training data
dataiter = iter(train_loader)
images, labels = next(dataiter)

for i in range(10):
    img = images[i].numpy().transpose((1, 2, 0))  # Convert from CHW to HWC
    axes[i].imshow(img)
    axes[i].set_title(f'{class_names[labels[i]]}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
import torch.nn as nn

# Define CNN model with 5 convolutional layers and BatchNormalization
class CNNModel(nn.Module):
    def __init__(self, num_classes=3):
        super(CNNModel, self).__init__()

        # Convolutional layers with BatchNorm
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) # must be same to the output channels always

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        # Activation and Pooling
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(512 * 1 * 1, 256)
        self.fc2 = nn.Linear(256, num_classes)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):

        x = self.pool(self.relu(self.bn1(self.conv1(x))))  # (Batch, 32, 16, 16)
        x = self.pool(self.relu(self.bn2(self.conv2(x))))  # (Batch, 64, 8, 8)
        x = self.pool(self.relu(self.bn3(self.conv3(x))))  # (Batch, 128, 4, 4
        x = self.pool(self.relu(self.bn4(self.conv4(x))))  # (Batch, 256, 2, 2)
        x = self.pool(self.relu(self.bn5(self.conv5(x))))  # (Batch, 512, 1, 1)
        x = x.view(x.size(0), -1)  # (Batch, 512*1*1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel(num_classes=num_classes).to(device)

print(f"Using device: {device}")
print(model)

In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval() # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
# Write your code here
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss() # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer
num_epochs = 5 # Number of epochs

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

print("Training Complete :)")

# Plot the training and validation losses and accuracy
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(range(1, num_epochs + 1), train_losses, label='Training Loss', marker='o')
axes[0].plot(range(1, num_epochs + 1), val_losses, label='Validation Loss', marker='o')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Plot accuracy
axes[1].plot(range(1, num_epochs + 1), train_accuracies, label='Training Accuracy', marker='o')
axes[1].plot(range(1, num_epochs + 1), val_accuracies, label='Validation Accuracy', marker='o')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
class CNNResidual(nn.Module):
    def __init__(self, num_classes=3):
        super(CNNResidual, self).__init__()

        # Convolutional layers with Batch Normalization (somehow similar to the one in U-Net lab)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
################################ skip from here ##################################
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)

        # Activation and Pooling
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.skip = nn.Sequential(
            nn.MaxPool2d(kernel_size=2, stride=2),  # Reduce spatial size once
            nn.Conv2d(64, 256, kernel_size=1),  # Match channels from 64 to 256 ;) as from layer 2 to layer 4
            nn.BatchNorm2d(256)
        )

        # Fully Connected Layers
        self.fc1 = nn.Linear(512 * 1 * 1, 256)
        self.fc2 = nn.Linear(256, num_classes)


    def forward(self, x):

        x = self.pool(self.relu(self.bn1(self.conv1(x))))  # (Batch, 32, 16, 16)

        # Conv block 2 - Save for skip connection
        x = self.relu(self.bn2(self.conv2(x)))
        skip = x  # Save the output of conv2 before pooling
        x = self.pool(x)  # (Batch, 64, 8, 8)

        # Conv block 3
        x = self.pool(self.relu(self.bn3(self.conv3(x))))  # (Batch, 128, 4, 4)

        # Conv block 4 - Add skip connection from conv2
        x = self.relu(self.bn4(self.conv4(x)))  # (Batch, 256, 4, 4)

        # Adapt skip connection to match dimensions
        skip = self.skip(skip) # (Batch, 256, 4, 4)

        #residual connection
        x = x + skip  # Element-wise addition
        x = self.pool(x)  # (Batch, 256, 2, 2)


        x = self.pool(self.relu(self.bn5(self.conv5(x))))  # (Batch, 512, 1, 1)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# Create residual model
model_res = CNNResidual(num_classes=num_classes).to(device)
print(model_res)

# Train the residual model
criterion_res = nn.CrossEntropyLoss()
optimizer_res = optim.Adam(model_res.parameters(), lr=0.001)
num_epochs_res = 10

# Lists to store metrics
train_losses_res = []
val_losses_res = []
train_accuracies_res = []
val_accuracies_res = []

# Training process
for epoch in range(num_epochs_res):
    train_loss, train_accuracy = train_one_epoch(model_res, train_loader, criterion_res, optimizer_res, device)
    val_loss, val_accuracy = validate(model_res, test_loader, criterion_res, device)

    # Store metrics
    train_losses_res.append(train_loss)
    val_losses_res.append(val_loss)
    train_accuracies_res.append(train_accuracy)
    val_accuracies_res.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs_res}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(range(1, num_epochs_res + 1), train_losses_res, label='Residual Train Loss', marker='o')
axes[0].plot(range(1, num_epochs_res + 1), val_losses_res, label='Residual Val Loss', marker='o')
axes[0].set_title('Residual Model: Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)
axes[1].plot(range(1, num_epochs_res + 1), train_accuracies_res, label='Residual Train Acc', marker='o')
axes[1].plot(range(1, num_epochs_res + 1), val_accuracies_res, label='Residual Val Acc', marker='o')
axes[1].set_title('Residual Model: Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()